In [5]:
"""
PIPELINE STAGE: Robust Data Extraction & Spatial Standardization (Consumption Metrics)
PERIOD: January 2020 - October 2021
LIBRARIES: os, pandas, lxml, zipfile, re

1. OBJECTIVE:
   Develop an automated recovery and extraction script to bypass CRC (Cyclic Redundancy Check) 
   errors in corrupted Word (.docx) files. Extract billed electricity consumption data by province, 
   standardize geographic names, and export to a strictly sorted, final structured database.

2. DIRECTORY STRUCTURE:
   - Source: os.path.join("..", "..", "raw_data", "energy_reports")
   - Output: os.path.join("..", "..", "processed_data", "final", "Electricity_Consumption_Final.xlsx")
   - System: Dynamically generates the output directory structure if it does not exist.

3. TECHNICAL WORKFLOW (RESILIENCE STRATEGY):
   A. Direct XML Parsing: Due to potential image-related CRC errors, bypass standard 'python-docx' 
      loaders. Access 'word/document.xml' via 'zipfile' and 'lxml' for raw data recovery.
   B. Contextual Table Identification: Locate tables by scanning preceding paragraphs for the 
      target caption (e.g., "Döneminde Faturalanan Elektrik Tüketiminin İl ve Tüketici Türü Bazında Dağılımı").
   C. Temporal Transformation: Parse string-based month names and years from filenames 
      (e.g., 'july_2020.docx') and map them into numerical values for time-series tracking.

4. DATA CLEANING & STANDARDIZATION RULES:
   A. Spatial Normalization (Regex): Standardize province names in Column 0 ('S_0') to ensure database integrity:
      - Matches containing "MARAŞ" -> "KAHRAMANMARAŞ"
      - Matches containing "AFYON" -> "AFYONKARAHİSAR"
      - Matches containing "HAKK" -> "HAKKARİ"
   B. Row Filtering: Remove non-data header/footer rows such as "İller", "Genel Toplam", and empty strings.
   C. Numerical Conversion: Process consumption data from the 7th column (Index 'S_6'). 
      Format Turkish numeric strings (e.g., 434.992,61) into numerical formats by handling decimals 
      and stripping thousand separators.

5. OUTPUT SPECIFICATIONS:
   - Final Schema: [Province, Year, Month, Consumption]
   - Sorting Logic: Ascending order primarily by 'Province', then 'Year', and 'Month'.
   - Logging: Real-time console reporting for processing status, missing files, and final row counts.
"""
import os
import re
import zipfile
import pandas as pd
from lxml import etree

# --- AYARLAR ---
BASE_PATH = os.path.join("..", "..", "raw_data", "energy_reports")
OUTPUT_TUKETICI = os.path.join("..", "..", "processed_data", "steps", "01_2020_2021_subscriber.xlsx")

GEÇERLİ_YILLAR = {
    "2020": list(range(1, 13)),
    "2021": list(range(1, 11))
}

# --- ANA DÖNGÜ ---
all_data = []
print("Veriler işleniyor, lütfen bekleyin...")

for year, months in GEÇERLİ_YILLAR.items():
    year_path = os.path.join(BASE_PATH, year)
    if not os.path.exists(year_path): continue

    for file_name in os.listdir(year_path):
        if not file_name.endswith(".docx"): continue
        
        parts = file_name.replace(".docx", "").lower().split("_")
        if len(parts) < 2: continue
        
        m_name, f_year = parts[0], parts[1]
        m_num = MONTH_MAP.get(m_name)
        
        if f_year == year and m_num in months:
            file_full_path = os.path.join(year_path, file_name)
            # HATA DÜZELTİLDİ: target_caption (küçük harf) kullanıldı
            table_rows = get_table_from_xml(file_full_path, target_caption)
            
            if table_rows:
                for row in table_rows:
                    all_data.append([year, m_num] + row)
                print(f"Başarılı: {file_name}")
            else:
                print(f"Hata/Bulunamadı: {file_name}")

# --- VERİ TEMİZLEME VE KAYIT ---
if all_data:
    # Çıktı klasörü yoksa oluştur (Hata almamak için eklendi)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    max_cols = max(len(r) for r in all_data)
    temp_headers = ["Year", "Month"] + [f"S_{i}" for i in range(max_cols - 2)]
    df = pd.DataFrame(all_data, columns=temp_headers)

    # 1. Şehir adlarını regex ile düzelt
    df["S_0"] = df["S_0"].replace(SEHIR_DUZELTME_REGEX, regex=True)

    # 2. 'İller' ve 'Genel Toplam' satırlarını sil
    df = df[~df["S_0"].isin(["İller", "Genel Toplam", ""])]

    # 3. Sayı formatını düzenle
    df["S_6"] = df["S_6"].apply(format_consumption)

    # 4. İstenen Sütunları Seç ve Başlıkları Düzenle
    df_final = df[["Year", "Month", "S_0", "S_6"]].copy()
    df_final.columns = ["Year", "Month","Province" , "Consumption"]
    df_final = df_final[["Province", "Year", "Month", "Consumption"]]
    df_final = df_final.sort_values(by=["Province", "Year", "Month"]).reset_index(drop=True)

    # 5. Excel'e Kaydet
    df_final.to_excel(output_path, index=False)
    
    print("-" * 40)
    print(f"İŞLEM TAMAMLANDI!")
    print(f"Dosya: {output_path}")
    print(f"Toplam Satır: {len(df_final)}")
else:
    print("Veri çekilemedi. Klasörleri veya dosya isimlerini kontrol edin.")

Veriler işleniyor, lütfen bekleyin...
Başarılı: April_2020.docx
Başarılı: August_2020.docx
Başarılı: December_2020.docx
Başarılı: February_2020.docx
Başarılı: January_2020.docx
Başarılı: July_2020.docx
Başarılı: June_2020.docx
Başarılı: March_2020.docx
Başarılı: May_2020.docx
Başarılı: November_2020.docx
Başarılı: October_2020.docx
Başarılı: September_2020.docx
Başarılı: April_2021.docx
Başarılı: August_2021.docx
Başarılı: February_2021.docx
Başarılı: January_2021.docx
Başarılı: July_2021.docx
Başarılı: June_2021.docx
Başarılı: March_2021.docx
Başarılı: May_2021.docx
Başarılı: October_2021.docx
Başarılı: September_2021.docx
----------------------------------------
İŞLEM TAMAMLANDI!
Dosya: ..\..\processed_data\steps\02_2020_2021_consumption.xlsx
Toplam Satır: 1782
